# 03 — Data Preprocessing

This notebook focuses on preparing the climate dataset for machine learning model development.

The preprocessing workflow will address data quality, missing values, temporal validation, feature selection, outlier investigation, feature engineering, and feature scaling.

All transformations will be designed to prevent data leakage and maintain consistency between training, validation, and testing datasets.

## 3.1 Environment Setup and Data Loading

This section configures the project environment and loads the integrated datasets generated during the dataset integration stage.

Existing reusable functions from the project source modules will be inspected and reused to maintain a modular and consistent preprocessing workflow.

### 3.1.1 Project Environment Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project configuration
from config import (
    TRAIN_DIR,
    TEST_DIR,
    DATA_DIR,
    MODELS_DIR,
    REPORTS_DIR
)

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Existing project functions
from src.data_loader import (
    load_and_standardize_datasets,
    merge_datasets
)

from src.feature_engineering import (
    add_lag_and_rolling_features,
    add_calendar_features,
    add_domain_informed_features,
)

print("Project root:", PROJECT_ROOT)
print("Environment setup completed.")

Project root: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge
Environment setup completed.


#### 3.1.2 Load Integrated Datasets

The standardized climate datasets will be loaded and merged using the reusable functions implemented in the project source module.

The original observations will remain unchanged at this stage. Preprocessing operations will be performed in subsequent sections.

In [2]:
# Load and standardize training datasets
train_datasets = load_and_standardize_datasets(TRAIN_DIR)

# Load and standardize testing datasets
test_datasets = load_and_standardize_datasets(TEST_DIR)

# Merge climate features
train_merged = merge_datasets(train_datasets)
test_merged = merge_datasets(test_datasets)

print("Training dataset shape:", train_merged.shape)
print("Testing dataset shape:", test_merged.shape)

Training dataset shape: (2214, 18)
Testing dataset shape: (738, 17)



## 3.2 Data Quality Assessment

This section examines data quality issues that require further investigation before model development.

The assessment focuses on missing value patterns and their potential impact on preprocessing and model evaluation.

Previously completed structural validation checks, including duplicate date verification and dataset alignment, are not repeated in this notebook.


### 3.2.1 Missing Values Investigation

This subsection examines missing values in the training and testing datasets.

The number and percentage of missing values are calculated for each feature to identify incomplete variables and determine the extent of missing data.

No imputation or deletion is performed at this stage.

In [3]:
# Calculate missing values in training data
train_missing = pd.DataFrame({
    "Missing Count": train_merged.isnull().sum(),
    "Missing Percentage (%)": (
        train_merged.isnull().mean() * 100
    )
})

# Calculate missing values in testing data
test_missing = pd.DataFrame({
    "Missing Count": test_merged.isnull().sum(),
    "Missing Percentage (%)": (
        test_merged.isnull().mean() * 100
    )
})

print("TRAINING DATASET — Missing Values")
display(train_missing.round(2))

print("TESTING DATASET — Missing Values")
display(test_missing.round(2))

TRAINING DATASET — Missing Values


,Missing Count,Missing Percentage (%)
date,0,0.0
5cm_soli_moist,0,0.0
mean_dew_point_temp,0,0.0
max_temp,0,0.0
sea_level_pressure,0,0.0
dryspell_warn_7d,0,0.0
sst_cameroon_mean_temperature_kelvin,0,0.0
sst_cameroon_mean_temperature_celsius,0,0.0
sst_cameroon_mean_temperature_uncertainty,0,0.0
sst_cameroon_fraction_of_sea_ice_covered_ocean,0,0.0


TESTING DATASET — Missing Values


,Missing Count,Missing Percentage (%)
date,0,0.00
5cm_soli_moist,0,0.00
mean_dew_point_temp,1,0.14
max_temp,1,0.14
sea_level_pressure,1,0.14
sst_cameroon_mean_temperature_kelvin,123,16.67
sst_cameroon_mean_temperature_celsius,123,16.67
sst_cameroon_mean_temperature_uncertainty,123,16.67
sst_cameroon_fraction_of_sea_ice_covered_ocean,123,16.67
sst_indian_ocean_mean_temperature_kelvin,123,16.67


#### 3.2.1.1 Investigating Missing Value Patterns

This subsection investigates the temporal distribution of missing values in the testing dataset.

The objective is to determine whether missing observations are isolated or concentrated within specific dates and groups of features.

Understanding these patterns is essential before selecting an appropriate missing value treatment strategy.

In [4]:
# Identify dates containing missing values
missing_dates = test_merged.loc[
    test_merged.isnull().any(axis=1),
    ["date"]
].copy()

# Add the number of missing features per date
missing_dates["missing_features"] = (
    test_merged.isnull().sum(axis=1)
    .loc[missing_dates.index]
)

print("Number of dates containing missing values:",
      len(missing_dates))

print("\nMissing dates and affected feature counts:")
display(missing_dates)

# Identify missing date ranges for each feature
for column in test_merged.columns:
    missing_mask = test_merged[column].isnull()

    if missing_mask.any():
        dates = test_merged.loc[missing_mask, "date"]

        print(
            f"{column}: "
            f"{dates.min().date()} to {dates.max().date()} "
            f"({missing_mask.sum()} missing values)"
        )

Number of dates containing missing values: 124

Missing dates and affected feature counts:


,date,missing_features
441,2023-09-11,6
615,2025-07-01,8
616,2025-07-02,8
617,2025-07-03,8
618,2025-07-04,8
...,...,...
733,2025-10-27,8
734,2025-10-28,8
735,2025-10-29,8
736,2025-10-30,8


mean_dew_point_temp: 2023-09-11 to 2023-09-11 (1 missing values)
max_temp: 2023-09-11 to 2023-09-11 (1 missing values)
sea_level_pressure: 2023-09-11 to 2023-09-11 (1 missing values)
sst_cameroon_mean_temperature_kelvin: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_mean_temperature_celsius: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_mean_temperature_uncertainty: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_fraction_of_sea_ice_covered_ocean: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_kelvin: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_celsius: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_uncertainty: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_fraction_of_sea_ice_covered_ocean: 2025-07-01 to 2025-10-31 (123 missing values)
potential_water_deficit: 2023-09-11 to 2023-09-11 (1 missing values)
surface_pressure: 2023-09-11

#### Key Findings

- A total of 124 dates contain missing values in the testing dataset.
- Six numerical features have one missing observation each, all occurring on September 11, 2023.
- All eight SST features have 123 missing observations each.
- SST missing values are concentrated in the continuous period from July 1, 2025, through October 31, 2025.
- The SST missingness pattern indicates a temporal data coverage gap rather than isolated missing observations.
- Missing values originate from two distinct patterns: a single-day data gap and a multi-month SST coverage gap.

No missing values are imputed or removed at this stage. Treatment decisions are deferred until the temporal validation strategy and modeling requirements are established.

## 3.3 Temporal Validation Strategy

This section establishes a time-aware validation strategy for model development.

Since the dataset represents chronological climate observations, validation must preserve temporal order to prevent data leakage and provide a realistic estimate of performance on future observations.

Candidate validation periods will be examined before selecting the final training and validation split.

### 3.3.1 Evaluating Candidate Validation Periods

This subsection evaluates different chronological holdout periods using the training dataset.

The comparison focuses on dataset size and positive-class representation in both training and validation periods.

The objective is to select a validation period that provides sufficient positive warning cases for meaningful model evaluation.

In [5]:
# Prepare chronological training data
temporal_data = train_merged[["date", "dryspell_warn_7d"]].copy()

temporal_data["year"] = temporal_data["date"].dt.year

last_year = temporal_data["year"].max()

candidate_periods = [2, 3, 4]

validation_results = []

for n_years in candidate_periods:
    validation_start_year = last_year - n_years + 1

    train_period = temporal_data[
        temporal_data["year"] < validation_start_year
    ]

    validation_period = temporal_data[
        temporal_data["year"] >= validation_start_year
    ]

    validation_results.append({
        "Validation Years": n_years,
        "Validation Start": validation_start_year,
        "Training Samples": len(train_period),
        "Training Positive Cases": train_period["dryspell_warn_7d"].sum(),
        "Validation Samples": len(validation_period),
        "Validation Positive Cases": validation_period["dryspell_warn_7d"].sum(),
        "Validation Warning Rate": validation_period["dryspell_warn_7d"].mean()
    })

validation_comparison = pd.DataFrame(validation_results)

display(validation_comparison)

,Validation Years,Validation Start,Training Samples,Training Positive Cases,Validation Samples,Validation Positive Cases,Validation Warning Rate
0,2,2018,1968,310,246,14,0.056911
1,3,2017,1845,296,369,28,0.075881
2,4,2016,1722,287,492,37,0.075203


#### Key Findings

- Three chronological validation periods were evaluated: 2, 3, and 4 years.
- The 2-year validation period (2018–2019) contains 14 positive cases.
- The 3-year validation period (2017–2019) contains 28 positive cases.
- The 4-year validation period (2016–2019) contains 37 positive cases.
- The 4-year validation period provides the largest number of positive cases for evaluating warning detection.
- The training dataset retains 287 positive cases when using the 4-year validation period.

The 2016–2019 period is considered the leading candidate for temporal validation, subject to further examination of the annual distribution of positive cases.

### 3.3.2 Annual Class Distribution in the Candidate Validation Period

This subsection examines the annual distribution of positive and negative target observations within the candidate validation period (2016–2019).

The analysis helps determine whether positive warning cases are reasonably represented across the validation years.

In [6]:
# Select the candidate validation period
candidate_validation = temporal_data[
    temporal_data["year"] >= 2016
].copy()

# Calculate annual class distribution
annual_validation_summary = (
    candidate_validation
    .groupby("year")["dryspell_warn_7d"]
    .agg(
        Total_Samples="count",
        Positive_Cases="sum",
        Warning_Rate="mean"
    )
)

annual_validation_summary["Negative_Cases"] = (
    annual_validation_summary["Total_Samples"]
    - annual_validation_summary["Positive_Cases"]
)

display(annual_validation_summary)

,Total_Samples,Positive_Cases,Warning_Rate,Negative_Cases
year,,,,
2016,123,9,0.073171,114
2017,123,14,0.113821,109
2018,123,14,0.113821,109
2019,123,0,0.000000,123


#### Key Findings

- The candidate validation period (2016–2019) contains 492 observations.
- A total of 37 positive warning cases are present in the validation period.
- Positive cases are distributed across three years:
  - 2016: 9 positive cases.
  - 2017: 14 positive cases.
  - 2018: 14 positive cases.
- No positive warning cases were recorded in 2019.
- The overall validation warning rate is approximately 7.52%.
- The candidate training period retains 287 positive cases.

The 2016–2019 period is selected as the candidate temporal validation period. The absence of positive cases in 2019 will be considered when interpreting model evaluation results.

## 3.4 Feature Availability and Data Leakage Assessment

This section examines whether the input features are available at the time a dry spell warning is expected to be generated.

The objective is to identify potential data leakage caused by using information that would not be available at prediction time.

Feature availability must be verified before finalizing the preprocessing and model development pipeline.

### 3.4.1 Target Variable Inspection

This subsection examines the target variable and its temporal distribution to understand how dry-spell warning labels are represented in the training dataset.

The inspection focuses on target values, class representation, and the relationship between warning labels and observation dates.

The competition dataset description does not provide the exact operational labeling formula or warning issuance timing. Therefore, these details will not be assumed without further evidence.

In [7]:
# Inspect target variable
target_summary = train_merged["dryspell_warn_7d"].value_counts(
    dropna=False
).sort_index()

print("Target values:")
display(target_summary)

print("\nTarget data type:")
print(train_merged["dryspell_warn_7d"].dtype)

print("\nTarget unique values:")
print(train_merged["dryspell_warn_7d"].unique())

print("\nTarget missing values:")
print(train_merged["dryspell_warn_7d"].isna().sum())

Target values:


dryspell_warn_7d
0    1890
1     324
Name: count, dtype: int64


Target data type:
int64

Target unique values:
[0 1]

Target missing values:
0


#### Key Findings

- The target variable contains 2,214 observations.
- Class 0 represents 1,890 observations.
- Class 1 represents 324 observations.
- The target variable uses the integer data type (`int64`).
- Only two unique values are present: 0 and 1.
- No missing values were found in the target variable.
- The target column satisfies the expected binary classification format.

The target variable is structurally valid for model development. However, its exact labeling methodology and warning issuance timing remain undocumented in the available dataset description.

### 3.4.2 Comparing Core Features with the Reference Solution

This subsection compares the nine core climate variables reported in the fourth-place competition solution with the features currently available in our integrated dataset.

The objective is to identify matching variables, naming differences, and missing variables before feature engineering.

No features will be added or removed based solely on this comparison.

In [8]:
# Core features reported in the fourth-place solution
reference_features = {
    "5cm_soil_moisture": "5cm_soli_moist",
    "potential_water_deficit": "potential_water_deficit",
    "vapor_pressure_deficit": "vapor_pressure_deficit",
    "u10_mean": None,
    "2m_temp": "2m_temp",
    "max_temp": "max_temp",
    "mean_dew_point_temp": "mean_dew_point_temp",
    "sea_level_pressure": "sea_level_pressure",
    "surface_pressure": "surface_pressure"
}

# Compare reference features with our integrated dataset
comparison_results = []

for reference_name, project_name in reference_features.items():
    available = (
        project_name is not None
        and project_name in train_merged.columns
    )

    comparison_results.append({
        "Reference Feature": reference_name,
        "Project Feature": project_name if project_name else "Not identified",
        "Available": available
    })

feature_comparison = pd.DataFrame(comparison_results)

display(feature_comparison)

,Reference Feature,Project Feature,Available
0,5cm_soil_moisture,5cm_soli_moist,True
1,potential_water_deficit,potential_water_deficit,True
2,vapor_pressure_deficit,vapor_pressure_deficit,True
3,u10_mean,Not identified,False
4,2m_temp,2m_temp,True
5,max_temp,max_temp,True
6,mean_dew_point_temp,mean_dew_point_temp,True
7,sea_level_pressure,sea_level_pressure,True
8,surface_pressure,surface_pressure,True


#### Key Findings

- Eight of the nine core climate features reported in the fourth-place solution are available in the integrated training dataset.
- The reference feature `5cm_soil_moisture` corresponds to `5cm_soli_moist` in the project dataset, with a difference in naming.
- The feature `u10_mean` is not available in the original dataset.
- The comparison confirms that the project dataset contains eight of the nine core features used in the reference solution.
- No additional features are introduced or removed based on this comparison.

The project will proceed using the available eight core features, along with the additional SST features already present in the dataset.

## 3.5 Missing Value Treatment

This section defines a missing value treatment strategy for the training and testing datasets.

The training dataset contains no missing values, while the testing dataset contains missing observations in several climate features.

Missing values will be handled using a strategy that avoids data leakage and preserves the original temporal structure.

### 3.5.1 Missing Value Treatment Strategy

The testing dataset contains two distinct missing value patterns:

- Isolated missing observations affecting six climate features on September 11, 2023.
- A continuous missing period affecting all eight SST features from July 1, 2025, to October 31, 2025.

Since these patterns differ in their temporal characteristics, the imputation strategy must be evaluated carefully.

No imputation will be applied until its impact can be assessed using the temporal validation dataset.

## 3.6 Feature Engineering

This section introduces temporal features to capture historical patterns in climate variables.

The feature engineering process focuses on lag features and rolling statistics derived from past observations.

All transformations will preserve chronological order and prevent information from future observations from entering the feature set.

### 3.6.1 Lag Features and Rolling Statistics

Lag features capture previous observations, while rolling statistics summarize recent climate conditions.

The following transformations will be applied to the available core climate features:

- Lag features: 1, 3, and 7 observations.
- Rolling mean features: 7 and 14 observations.

All calculations will be performed separately for each year to prevent temporal features from crossing seasonal data gaps.

Rolling statistics will use shifted observations to ensure that only past values contribute to each feature.

In [9]:
# Core climate features available in our dataset
core_features = [
    "5cm_soli_moist",
    "potential_water_deficit",
    "vapor_pressure_deficit",
    "2m_temp",
    "max_temp",
    "mean_dew_point_temp",
    "sea_level_pressure",
    "surface_pressure"
]

# Apply temporal feature engineering
train_engineered = add_lag_and_rolling_features(
    train_merged,
    core_features
)

test_engineered = add_lag_and_rolling_features(
    test_merged,
    core_features
)

# Display dataset dimensions
print("Original training shape:", train_merged.shape)
print("Engineered training shape:", train_engineered.shape)

print("\nOriginal testing shape:", test_merged.shape)
print("Engineered testing shape:", test_engineered.shape)

Original training shape: (2214, 18)
Engineered training shape: (2214, 58)

Original testing shape: (738, 17)
Engineered testing shape: (738, 57)


#### Key Findings

- Temporal feature engineering successfully generated 40 additional features from eight core climate variables.
- Three lag features (1, 3, and 7 observations) were created for each variable.
- Two rolling mean features (7 and 14 observations) were created for each variable.
- The training dataset now contains 58 columns, compared to 18 originally.
- The testing dataset now contains 57 columns, compared to 17 originally.
- No observations were removed during feature engineering.

The engineered datasets preserve the original chronological observations while incorporating historical climate information.

### 3.6.2 Inspecting Missing Values After Feature Engineering

This subsection examines missing values introduced by temporal feature engineering.

The analysis identifies missing values in the newly generated lag and rolling features, while distinguishing them from missing values already present in the original datasets.

In [10]:
# Identify missing values in engineered datasets
for name, df in {
    "Training": train_engineered,
    "Testing": test_engineered
}.items():

    missing_summary = df.isnull().sum()
    missing_summary = missing_summary[missing_summary > 0]

    print(f"\n{name} - Missing Values:")
    display(missing_summary.to_frame("Missing Count"))

    print("Total missing values:", df.isnull().sum().sum())


Training - Missing Values:


,Missing Count
5cm_soli_moist_lag_1,18
5cm_soli_moist_lag_3,54
5cm_soli_moist_lag_7,126
5cm_soli_moist_rolling_mean_7,126
5cm_soli_moist_rolling_mean_14,252
potential_water_deficit_lag_1,18
potential_water_deficit_lag_3,54
potential_water_deficit_lag_7,126
potential_water_deficit_rolling_mean_7,126
potential_water_deficit_rolling_mean_14,252


Total missing values: 4608

Testing - Missing Values:


,Missing Count
mean_dew_point_temp,1
max_temp,1
sea_level_pressure,1
sst_cameroon_mean_temperature_kelvin,123
sst_cameroon_mean_temperature_celsius,123
sst_cameroon_mean_temperature_uncertainty,123
sst_cameroon_fraction_of_sea_ice_covered_ocean,123
sst_indian_ocean_mean_temperature_kelvin,123
sst_indian_ocean_mean_temperature_celsius,123
sst_indian_ocean_mean_temperature_uncertainty,123


Total missing values: 2670


#### Key Findings

- Temporal feature engineering introduced missing values at the beginning of each yearly observation period.
- Lag features generated missing values according to their respective lag periods.
- Rolling mean features generated missing values until sufficient historical observations became available.
- The training dataset contains 4,608 missing values, all introduced by temporal feature engineering.
- The testing dataset contains 2,670 missing values, resulting from both original data gaps and temporal feature engineering.
- Missing values in the original testing features also propagate into subsequent lag and rolling calculations.

These results confirm that the generated missing values are primarily associated with the temporal feature construction process and the previously identified testing data gaps.

### 3.6.3 Calendar Features

Calendar features capture the seasonal position of each observation.

The following features will be extracted from the date column:

- Month.
- Day of year.
- Cyclical day-of-year encoding using sine and cosine transformations.

Cyclical encoding represents the continuity of the calendar and helps models capture seasonal patterns without treating the end and beginning of the year as distant points.

In [11]:

# Apply calendar feature engineering
train_engineered = add_calendar_features(train_engineered)
test_engineered = add_calendar_features(test_engineered)

# Display dataset dimensions
print("Training shape:", train_engineered.shape)
print("Testing shape:", test_engineered.shape)

# Preview calendar features
display(
    train_engineered[
        [
            "date",
            "month",
            "day_of_year",
            "day_of_year_sin",
            "day_of_year_cos"
        ]
    ].head()
)


Training shape: (2214, 62)
Testing shape: (738, 61)


,date,month,day_of_year,day_of_year_sin,day_of_year_cos
0,2002-07-01,7,182,0.010751,-0.999942
1,2002-07-02,7,183,-0.006451,-0.999979
2,2002-07-03,7,184,-0.023651,-0.999720
3,2002-07-04,7,185,-0.040844,-0.999166
4,2002-07-05,7,186,-0.058026,-0.998315


#### Key Findings

- Four calendar features were successfully generated: month, day of year, and cyclical sine and cosine encodings.
- The training dataset now contains 62 columns.
- The testing dataset now contains 61 columns.
- No observations were removed during calendar feature engineering.

Calendar features were added successfully to represent seasonal timing and cyclical patterns.


### 3.6.4 Domain-Informed Climate Features

In this section, we create additional climate features based on domain knowledge.

The features capture recent changes in soil moisture and water deficit, the temperature-dew point spread, and the sea surface temperature contrast between the Indian Ocean and Cameroon.

These features are derived from the available dataset variables.


In [12]:
train_engineered = add_domain_informed_features(train_engineered)
test_engineered = add_domain_informed_features(test_engineered)

print("Train shape:", train_engineered.shape)
print("Test shape:", test_engineered.shape)

Train shape: (2214, 66)
Test shape: (738, 65)


## 3.7 Modeling Dataset Preparation

This section prepares the engineered datasets for machine learning.

The data will be split chronologically to simulate forecasting on unseen future observations:

- Training period: 2002–2015.
- Validation period: 2016–2019.
- Testing period: 2020–2025.

The target variable will be separated from the input features.

The testing dataset will remain untouched during model training and validation. The same feature columns and ordering will be used across all datasets.

### 3.7.1 Temporal Split and Feature-Target Separation

The training and validation datasets will be created using the observation date.

The target variable will be separated from the feature matrix, and the date and target columns will be excluded from the input features.

Feature columns will be aligned across training, validation, and testing datasets to ensure consistent model inputs.

In [13]:
# Define temporal split
train_mask = train_engineered["date"].dt.year <= 2015
validation_mask = train_engineered["date"].dt.year.between(2016, 2019)

# Separate target from features
target_column = "dryspell_warn_7d"

feature_columns = [
    column
    for column in train_engineered.columns
    if column not in ["date", target_column]
]

# Create training and validation sets
X_train = train_engineered.loc[train_mask, feature_columns].copy()
y_train = train_engineered.loc[train_mask, target_column].copy()

X_val = train_engineered.loc[validation_mask, feature_columns].copy()
y_val = train_engineered.loc[validation_mask, target_column].copy()

# Prepare test features using the same feature columns
X_test = test_engineered[feature_columns].copy()

# Display dataset dimensions
print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("\nValidation features:", X_val.shape)
print("Validation target:", y_val.shape)

print("\nTesting features:", X_test.shape)

print("\nTraining positive cases:", int(y_train.sum()))
print("Validation positive cases:", int(y_val.sum()))

print("\nFeature columns aligned:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))

Training features: (1722, 64)
Training target: (1722,)

Validation features: (492, 64)
Validation target: (492,)

Testing features: (738, 64)

Training positive cases: 287
Validation positive cases: 37

Feature columns aligned: True


#### Key Findings

- The chronological split was successfully applied.
- The training dataset contains 1,722 observations from 2002–2015, including 287 positive cases.
- The validation dataset contains 492 observations from 2016–2019, including 37 positive cases.
- The testing dataset contains 738 observations from 2020–2025.
- All datasets contain 64 aligned input features with consistent column ordering.
- The target variable and date column were excluded from the feature matrices.

The temporal split preserves the forecasting setup and keeps the testing dataset separate from model development.

### 3.7.2 Save Prepared Modeling Data

The prepared training, validation, and testing datasets are saved for reuse in the model training stage.

This avoids repeating feature engineering and temporal splitting in the modeling notebook.

In [14]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

prepared_data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
}

pd.to_pickle(prepared_data, processed_dir / "modeling_data.pkl")

print("Prepared modeling data saved successfully.")
print(f"Location: {processed_dir.resolve()}")

Prepared modeling data saved successfully.
Location: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge\data\processed
